# 🧠 Chatbot de Expertos Temáticos con Ollama (modelo local `gemma3:1b`)

**Reto proyecto — Desarrollo de Soluciones IA**

Chatbot de consola que permite conversar con tres **expertos temáticos** — Programación, Marketing y Jurídico-Legal — usando el modelo **`gemma3:1b`** ejecutado **localmente con Ollama**, **100 % offline**. Mantiene historial de conversación multi-turno, permite **cambiar de experto** (conservando o reiniciando el historial), **reiniciar** o **salir**, e incluye **manejo de errores** de conexión/modelo.

### Mapeo de la estructura de archivos pedida → celdas de este notebook

```
├── main.py                  → Celda «Interfaz CLI» (run_cli)
├── experts/
│   ├── __init__.py          → (implícito)
│   └── expert_prompts.py    → Celda «Prompts de expertos»
├── core/
│   ├── __init__.py          → (implícito)
│   ├── chatbot.py           → Celda «Lógica del chatbot»
│   └── conversation.py      → Celda «Gestión del historial»
├── requirements.txt         → Celda de dependencias
└── README.md                → Este encabezado
```

### Configuración de Ollama (solo una vez, requiere internet)

1. **Instala la aplicación Ollama** desde <https://ollama.com/download> e instálala. ⚠️ Esto es distinto del paquete `pip install ollama`: la app es el **servidor**; el paquete pip es solo el **cliente** de Python.

2. **Arranca el servicio** (si no arrancó solo tras instalar). En una terminal (PowerShell/CMD en Windows, o el terminal en macOS/Linux):
   ```bash
   ollama serve
   ```
   En Windows normalmente queda corriendo en segundo plano (icono en la bandeja del sistema), así que este paso suele ser automático.

3. **Verifica que el servicio responde:**
   ```bash
   ollama list
   ```
   Si te devuelve una lista (aunque esté vacía), el servidor está activo. Si da error de conexión, vuelve al paso 2 y ejecuta `ollama serve`.
   - *Comprobación alternativa:* abre <http://localhost:11434> en el navegador. Si Ollama está corriendo verás el texto **"Ollama is running"**.

4. **Descarga el modelo** `gemma3:1b`:
   ```bash
   ollama pull gemma3:1b
   ```
   Para confirmar que quedó descargado, vuelve a ejecutar `ollama list` y comprueba que aparece `gemma3:1b`.

5. **Ejecuta este notebook** de arriba abajo. La celda de comprobación (sección 2) te confirmará el estado.

> Tras descargar el modelo, **todo funciona sin conexión a internet**: Ollama corre en tu máquina y `gemma3:1b` queda guardado en local. Lo único que necesita red es el `pip install` y el `ollama pull` iniciales.


## 1. Dependencias (`requirements.txt`)

```text
ollama
```

Ejecuta la siguiente celda una sola vez para instalar el SDK oficial de Ollama para Python.


In [10]:
# Instalación del SDK oficial de Ollama (ejecutar una vez)
%pip install -q ollama


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Comprobación del servicio y del modelo

Antes de chatear conviene verificar dos cosas: que **Ollama está en marcha** y que el modelo **`gemma3:1b` está descargado** localmente. Esta función devuelve un diagnóstico claro y se reutiliza en la CLI para informar del estado del sistema.


In [11]:
import os
from ollama import Client

# Modelo configurable mediante la variable de entorno OLLAMA_MODEL,
# con "gemma3:1b" como valor por defecto. Así puedes probar otros modelos
# locales sin tocar el código:
#   Windows (PowerShell):  $env:OLLAMA_MODEL = "llama3.2:1b"
#   macOS/Linux:           export OLLAMA_MODEL=llama3.2:1b
# También puedes pasarlo como argumento:  run_cli(modelo="llama3.2:1b")
MODELO = os.getenv("OLLAMA_MODEL", "gemma3:1b")


def comprobar_ollama(modelo: str = MODELO, host: str | None = None):
    """Comprueba que el servicio Ollama responde y que el modelo está disponible.

    Devuelve (ok: bool, mensaje: str). No requiere conexión a internet:
    solo habla con el servicio local de Ollama.
    """
    client = Client(host=host) if host else Client()

    # 1) ¿Está el servicio de Ollama en marcha?
    try:
        respuesta = client.list()
    except Exception as e:
        return False, (
            f"❌ No se pudo conectar con Ollama ({type(e).__name__}). "
            "Asegúrate de que está instalado y de que 'ollama serve' está en marcha. "
            "Descarga: https://ollama.com/download"
        )

    # 2) ¿Está descargado el modelo? (admite respuesta tipo objeto o dict)
    modelos = getattr(respuesta, "models", None)
    if modelos is None and isinstance(respuesta, dict):
        modelos = respuesta.get("models", [])
    nombres = []
    for m in modelos or []:
        nombre = getattr(m, "model", None)
        if nombre is None and isinstance(m, dict):
            nombre = m.get("model") or m.get("name")
        if nombre:
            nombres.append(nombre)

    disponible = any(n == modelo or n.startswith(modelo) for n in nombres)
    if not disponible:
        return False, (
            f"⚠️  Ollama responde, pero el modelo '{modelo}' no está descargado.\n"
            f"   Modelos locales: {', '.join(nombres) if nombres else '(ninguno)'}\n"
            f"   Descárgalo con:  ollama pull {modelo}"
        )

    return True, f"✅ Ollama operativo y modelo '{modelo}' disponible en local."


# Comprobación informativa al ejecutar la celda
ok, mensaje = comprobar_ollama()
print(mensaje)


✅ Ollama operativo y modelo 'gemma3:1b' disponible en local.


### 🛠️ Si ves `ConnectError` / `WinError 10061` (conexión rechazada)

Si la comprobación de arriba muestra algo como:

```
ConnectError: [WinError 10061] No se puede establecer una conexión ya que
el equipo de destino denegó expresamente dicha conexión
```

**no es un fallo del código**: es el manejo de errores funcionando. Significa que **el servicio de Ollama no está en marcha**. El SDK intenta conectarse a `http://localhost:11434` (donde escucha Ollama) y no hay nada escuchando ahí.

> ⚠️ Ojo: `pip install ollama` (la celda de dependencias) **solo instala la librería cliente de Python**; **no arranca el servidor**. El servidor es la aplicación Ollama, un programa aparte.

**Cómo solucionarlo (Windows):**

1. **Instala la aplicación Ollama** (no solo el paquete pip) desde <https://ollama.com/download>. Tras instalarla suele quedar corriendo en segundo plano (icono en la bandeja del sistema).
2. **Comprueba que el servicio responde.** En PowerShell o CMD:
   ```bash
   ollama list
   ```
   Si devuelve una lista (aunque esté vacía), el servidor está activo. Si da error de conexión, arráncalo con `ollama serve` y deja esa terminal abierta.
3. **Descarga el modelo** en otra terminal:
   ```bash
   ollama pull gemma3:1b
   ```
4. **Reejecuta la celda anterior** (`comprobar_ollama()`). Debe aparecer `✅ Ollama operativo y modelo 'gemma3:1b' disponible en local`.

**Verificación rápida:** abre <http://localhost:11434> en el navegador. Si Ollama está corriendo, verás el texto *"Ollama is running"*.

> ✅ De cara a la entrega, este error capturado, el mensaje claro y el hecho de que el historial quede limpio demuestran que el criterio de *manejo de errores para conexión o modelo no disponible* está cubierto. Cuando arranques Ollama, las celdas mostrarán además las respuestas reales de los tres expertos.


## 3. Prompts de expertos (`experts/expert_prompts.py`)

Cada experto se define con un **prompt de sistema** específico que modifica el comportamiento y el estilo del modelo. Se guardan en un diccionario `EXPERTOS` con una clave interna, un nombre visible y el prompt. Los tres están claramente diferenciados en **rol, ámbito, estilo** y en qué hacer cuando la pregunta queda fuera de su especialidad.


In [12]:
EXPERTOS = {
    "programacion": {
        "nombre": "💻 Experto en Programación",
        "prompt": (
            "Eres un ingeniero de software senior con amplia experiencia en desarrollo, "
            "arquitectura de software y buenas prácticas. Respondes SIEMPRE en español.\n"
            "Tu especialidad: diseño de sistemas, patrones de diseño, código limpio, testing, "
            "rendimiento, control de versiones y elección de tecnologías.\n"
            "Estilo: técnico y preciso. Explica el PORQUÉ de cada decisión y menciona los "
            "compromisos (trade-offs). Cuando aporte valor, incluye fragmentos de código bien "
            "comentados y nombra las mejores prácticas aplicables.\n"
            "Nivel de detalle: adáptalo al usuario. Si parece principiante, explica los conceptos "
            "base y evita la jerga; si parece avanzado, sé más conciso y entra en matices "
            "técnicos. Si no está claro, asume un nivel intermedio o pregúntalo.\n"
            "Formato de respuesta: explicación breve, luego un fragmento de código comentado "
            "(cuando aplique) y, al final, una lista de buenas prácticas.\n"
            "Ejemplo de estilo de respuesta:\n"
            "Pregunta: ¿lista o tupla en Python?\n"
            "Respuesta: Usa tupla para datos inmutables (más ligera y segura) y lista si vas a "
            "modificarlos; p. ej. punto = (3, 4) frente a colores = ['rojo']. "
            "Buena práctica: elige el tipo por su mutabilidad, no por costumbre.\n"
            "Si la pregunta NO es de programación, indícalo brevemente y sugiere cambiar al "
            "experto adecuado. Sé claro y conciso."
        ),
    },
    "marketing": {
        "nombre": "📈 Experto en Marketing",
        "prompt": (
            "Eres un estratega de marketing con experiencia en branding, posicionamiento, "
            "análisis de mercado y estrategias comerciales. Respondes SIEMPRE en español.\n"
            "Tu especialidad: público objetivo, propuesta de valor, embudos de conversión, "
            "canales y campañas, y métricas clave (CAC, LTV, ROI, tasa de conversión).\n"
            "Estilo: orientado a resultados y persuasivo, pero siempre apoyado en datos y "
            "razonamiento. Propón acciones CONCRETAS y MEDIBLES, no generalidades.\n"
            "Nivel de detalle: adáptalo al usuario. Si parece principiante, explica los términos "
            "(CAC, LTV, embudo...) con ejemplos sencillos; si parece avanzado, ve al grano con "
            "tácticas y números. Si no está claro, asume un nivel intermedio o pregúntalo.\n"
            "Formato de respuesta: usa listas numeradas de acciones y, junto a cada una, "
            "indica la métrica con la que se mediría (p. ej. ROI, CAC, tasa de conversión).\n"
            "Ejemplo de estilo de respuesta:\n"
            "Pregunta: ¿cómo capto a mis primeros clientes?\n"
            "Respuesta: 1) Define el público objetivo (métrica: leads cualificados). "
            "2) Lanza una campaña en el canal donde está ese público (métrica: CAC). "
            "3) Mide la conversión y ajusta (métrica: tasa de conversión).\n"
            "Si la pregunta NO es de marketing, indícalo brevemente y sugiere cambiar al "
            "experto adecuado. Sé práctico y directo."
        ),
    },
    "legal": {
        "nombre": "⚖️ Experto Jurídico-Legal",
        "prompt": (
            "Eres un asesor jurídico-legal especializado en normativa, contratos y cumplimiento. "
            "Respondes SIEMPRE en español.\n"
            "Tu especialidad: interpretación de normas, cláusulas contractuales, protección de "
            "datos, propiedad intelectual y obligaciones legales.\n"
            "Estilo: formal, prudente y estructurado. Distingue entre la regla general y lo que "
            "depende de la jurisdicción concreta; si falta contexto, pídelo.\n"
            "Nivel de detalle: adáptalo al usuario. Si parece lego en la materia, explica los "
            "términos jurídicos en lenguaje llano; si parece profesional, usa la terminología "
            "precisa. Si no está claro, asume un nivel intermedio o pregúntalo.\n"
            "Formato de respuesta: estructura por apartados (p. ej. 'Marco normativo', "
            "'Cláusulas clave', 'Riesgos' o 'Recomendaciones').\n"
            "Ejemplo de estilo de respuesta:\n"
            "Pregunta: ¿necesito contrato por escrito con un freelance?\n"
            "Respuesta: Marco normativo: el contrato verbal suele ser válido, pero el escrito "
            "aporta seguridad probatoria. Cláusulas clave: objeto, precio, plazos y "
            "confidencialidad. Riesgos: sin documento es difícil reclamar incumplimientos.\n"
            "IMPORTANTE: termina SIEMPRE recordando que tu respuesta es informativa y no "
            "sustituye el asesoramiento de un profesional colegiado.\n"
            "Si la pregunta NO es jurídica, indícalo brevemente y sugiere cambiar al experto "
            "adecuado."
        ),
    },
}


def listar_expertos() -> None:
    """Muestra los expertos disponibles con su número de selección."""
    for i, (clave, datos) in enumerate(EXPERTOS.items(), start=1):
        print(f"  {i}. {datos['nombre']}  (clave: {clave})")


## 4. Gestión del historial (`core/conversation.py`)

Clase `Conversation` que guarda el historial usuario↔asistente **en memoria** (sin base de datos). Mantiene un formato `{"role": "user"|"assistant", "content": str}` directamente compatible con la API de chat de Ollama. El *prompt de sistema* NO se guarda aquí: lo añade el chatbot en cada llamada según el experto activo, de modo que cambiar de experto no contamina el historial.


In [13]:
class Conversation:
    """Mantiene el historial de la conversación en memoria.

    max_mensajes: si se indica, conserva solo los últimos N mensajes para no
    saturar el contexto del modelo en conversaciones muy largas (None = sin límite).
    """

    def __init__(self, max_mensajes: int | None = None):
        self._messages: list[dict] = []
        self.max_mensajes = max_mensajes

    def _recortar(self) -> None:
        if self.max_mensajes is not None and len(self._messages) > self.max_mensajes:
            # Conserva únicamente los mensajes más recientes
            self._messages = self._messages[-self.max_mensajes:]

    def add_user(self, content: str) -> None:
        self._messages.append({"role": "user", "content": content})
        self._recortar()

    def add_assistant(self, content: str) -> None:
        self._messages.append({"role": "assistant", "content": content})
        self._recortar()

    def remove_last(self) -> None:
        """Elimina el último mensaje (útil si una llamada falla a medias)."""
        if self._messages:
            self._messages.pop()

    @property
    def messages(self) -> list[dict]:
        """Copia del historial en formato role/content."""
        return list(self._messages)

    def turnos(self) -> list[dict]:
        """Devuelve el historial agrupado por turnos {usuario, asistente}.

        Útil para logs o para exportar la conversación de forma legible.
        """
        pares: list[dict] = []
        pendiente = None
        for m in self._messages:
            if m["role"] == "user":
                pendiente = {"usuario": m["content"], "asistente": None}
                pares.append(pendiente)
            else:  # assistant
                if pendiente is not None and pendiente["asistente"] is None:
                    pendiente["asistente"] = m["content"]
                else:
                    pares.append({"usuario": None, "asistente": m["content"]})
                pendiente = None
        return pares

    def clear(self) -> None:
        self._messages.clear()

    def __len__(self) -> int:
        return len(self._messages)

    def __repr__(self) -> str:
        return f"Conversation({len(self._messages)} mensajes)"


## 5. Lógica del chatbot (`core/chatbot.py`)

La clase `Chatbot`:

1. Se conecta al servicio local de Ollama mediante el **SDK oficial** (`ollama.Client`).
2. Usa el modelo **`gemma3:1b`** en **modo streaming** (`client.chat(..., stream=True)`).
3. Construye cada petición como `[system del experto] + historial + nuevo mensaje`, conservando así el **contexto**.
4. Permite **cambiar de experto** conservando o reiniciando el historial, y **reiniciar** la conversación.
5. **Maneja errores** de conexión (servicio caído) y de modelo no disponible (`ResponseError`), informando con claridad y sin romper el programa.


In [14]:
from ollama import Client, ResponseError


class Chatbot:
    """Orquesta la conversación con un experto activo usando un modelo local de Ollama.

    Mantiene un historial INDEPENDIENTE por experto: al cambiar de experto y
    conservar el historial, cada uno recupera su propio contexto sin mezclarse.
    """

    def __init__(self, expertos: dict, modelo: str = MODELO,
                 experto_inicial: str | None = None, host: str | None = None,
                 max_mensajes: int | None = None):
        self.expertos = expertos
        self.modelo = modelo
        self.host = host
        self.client = Client(host=host) if host else Client()
        self.max_mensajes = max_mensajes
        # Un historial separado por cada clave de experto
        self.conversaciones = {
            clave: Conversation(max_mensajes=max_mensajes) for clave in expertos
        }
        self.experto_actual = experto_inicial or next(iter(expertos))

    # --- Información del experto activo ---
    @property
    def conversation(self) -> Conversation:
        """Historial del experto activo (cada experto tiene el suyo propio)."""
        return self.conversaciones[self.experto_actual]

    @property
    def system_prompt(self) -> str:
        return self.expertos[self.experto_actual]["prompt"]

    @property
    def nombre_experto(self) -> str:
        return self.expertos[self.experto_actual]["nombre"]

    # --- Disponibilidad del modelo (reutiliza comprobar_ollama) ---
    def comprobar_disponibilidad(self, mostrar: bool = True) -> bool:
        """Re-verifica que Ollama y el modelo siguen disponibles.

        Útil antes de una conversación larga por si el servicio se cae o el
        modelo se borra durante la ejecución.
        """
        ok, mensaje = comprobar_ollama(self.modelo, self.host)
        if mostrar:
            print(mensaje)
        return ok

    # --- Gestión de experto / historial ---
    def cambiar_experto(self, clave: str, conservar_historial: bool = False) -> None:
        if clave not in self.expertos:
            raise KeyError(f"Experto desconocido: {clave}")
        self.experto_actual = clave
        # Cada experto conserva su propio historial; si no se quiere conservar,
        # se reinicia ÚNICAMENTE el de este experto.
        if not conservar_historial:
            self.conversaciones[clave].clear()

    def reiniciar(self, todos: bool = False) -> None:
        """Reinicia el historial del experto actual (o de todos si todos=True)."""
        if todos:
            for conv in self.conversaciones.values():
                conv.clear()
        else:
            self.conversaciones[self.experto_actual].clear()

    # --- Conversación ---
    def enviar(self, texto_usuario: str) -> str | None:
        """Envía un mensaje al experto activo y devuelve la respuesta (o None si falla)."""
        self.conversation.add_user(texto_usuario)

        # system del experto + su historial => mantiene el contexto multi-turno
        mensajes = [{"role": "system", "content": self.system_prompt}]
        mensajes += self.conversation.messages

        partes: list[str] = []
        try:
            stream = self.client.chat(model=self.modelo, messages=mensajes, stream=True)
            for chunk in stream:
                # El SDK puede devolver objetos o dicts; soportamos ambos
                msg = getattr(chunk, "message", None)
                if msg is None and isinstance(chunk, dict):
                    msg = chunk.get("message")
                contenido = ""
                if msg is not None:
                    contenido = getattr(msg, "content", None)
                    if contenido is None and isinstance(msg, dict):
                        contenido = msg.get("content", "")
                if contenido:
                    print(contenido, end="", flush=True)
                    partes.append(contenido)
            print()

        except ResponseError as e:
            # Error devuelto por el servidor de Ollama (p. ej. modelo no descargado -> 404)
            self.conversation.remove_last()  # deshace el mensaje de usuario para poder reintentar
            if getattr(e, "status_code", None) == 404:
                print(f"\n❌ El modelo '{self.modelo}' no está disponible. "
                      f"Descárgalo con:  ollama pull {self.modelo}")
            else:
                print(f"\n❌ Error de Ollama ({getattr(e, 'status_code', '?')}): "
                      f"{getattr(e, 'error', e)}")
            return None

        except Exception as e:
            # Fallo de conexión (servicio caído) u otro problema inesperado
            self.conversation.remove_last()
            print(f"\n❌ No se pudo contactar con Ollama ({type(e).__name__}: {e}). "
                  "¿Está 'ollama serve' en marcha?")
            return None

        respuesta = "".join(partes)
        self.conversation.add_assistant(respuesta)
        return respuesta


## 6. Interfaz CLI (`main.py`)

Bucle de consola intuitivo. Muestra en todo momento el **experto activo** y las **opciones disponibles**. Comandos:

| Comando | Acción |
|---|---|
| `/cambiar` | Cambiar de experto (pregunta si conservar o reiniciar el historial) |
| `/reiniciar` | Borrar el historial del experto actual |
| `/experto` | Mostrar el experto activo |
| `/ayuda` | Mostrar la ayuda |
| `/salir` | Terminar el programa |

Cualquier otro texto se envía como mensaje al experto activo.

**Configuración opcional:** el modelo se puede cambiar con la variable de entorno `OLLAMA_MODEL` o con el argumento `run_cli(modelo="...")` (por defecto `gemma3:1b`). Y con `run_cli(max_mensajes=N)` se limita el historial a los últimos N mensajes para no saturar el contexto en conversaciones largas.

In [15]:
def _elegir_experto(actual: str | None = None) -> str:
    """Muestra el menú de expertos y devuelve la clave elegida."""
    claves = list(EXPERTOS.keys())
    print("\nExpertos disponibles:")
    listar_expertos()
    while True:
        sel = input("Elige un experto (número o clave): ").strip().lower()
        if sel in EXPERTOS:
            return sel
        if sel.isdigit() and 1 <= int(sel) <= len(claves):
            return claves[int(sel) - 1]
        if not sel and actual:
            return actual  # Enter mantiene el actual
        print("  Opción no válida, inténtalo de nuevo.")


def _mostrar_ayuda() -> None:
    print("\nComandos disponibles:")
    print("  /cambiar    cambiar de experto (cada experto guarda su propio historial)")
    print("  /reiniciar  borrar el historial del experto actual")
    print("  /comprobar  re-verificar que Ollama y el modelo siguen disponibles")
    print("  /experto    ver el experto activo y el tamaño del historial")
    print("  /ayuda      mostrar esta ayuda")
    print("  /salir      terminar")
    print("Escribe cualquier otro texto para enviarlo como mensaje al experto activo.")


def run_cli(modelo: str = MODELO, host: str | None = None,
            max_mensajes: int | None = None) -> None:
    """Bucle principal de la interfaz de línea de comandos.

    modelo:        nombre del modelo de Ollama (por defecto, la variable MODELO).
    host:          URL del servicio Ollama (None = localhost por defecto).
    max_mensajes:  límite opcional de mensajes por experto (None = sin límite).
    """
    print("=" * 64)
    print("🧠 Chatbot de expertos con Ollama (modelo local)")
    print("=" * 64)

    # Estado del sistema: conexión y modelo
    ok, mensaje = comprobar_ollama(modelo, host)
    print(mensaje)
    if not ok:
        print("\nResuelve el problema indicado y vuelve a ejecutar run_cli().")
        return

    bot = Chatbot(EXPERTOS, modelo=modelo, host=host, max_mensajes=max_mensajes)
    bot.experto_actual = _elegir_experto()

    print(f"\n▶️  Experto activo: {bot.nombre_experto}")
    _mostrar_ayuda()  # resumen de comandos al inicio de la sesión

    vacias = 0  # cuenta de líneas vacías consecutivas
    while True:
        try:
            entrada = input(f"\n[Tú · {bot.nombre_experto}] > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 ¡Hasta luego!")
            break

        if not entrada:
            vacias += 1
            if vacias >= 2:
                print("💡 Escribe una pregunta para el experto, o un comando como /ayuda.")
                vacias = 0
            continue
        vacias = 0

        comando = entrada.lower()

        if comando in ("/salir", "/exit", "/quit"):
            print("👋 ¡Hasta luego!")
            break

        if comando in ("/ayuda", "/help"):
            _mostrar_ayuda()
            continue

        if comando in ("/comprobar", "/check"):
            bot.comprobar_disponibilidad()
            continue

        if comando == "/experto":
            print(f"ℹ️  Experto activo: {bot.nombre_experto}  "
                  f"({len(bot.conversation)} mensajes en su historial)")
            continue

        if comando in ("/reiniciar", "/reset"):
            bot.reiniciar()
            print(f"🔄 Historial reiniciado. Sigues con {bot.nombre_experto}.")
            _mostrar_ayuda()  # recordatorio para quien se incorpora tarde
            continue

        if comando in ("/cambiar", "/change"):
            nueva_clave = _elegir_experto(actual=bot.experto_actual)
            conservar = input("¿Conservar el historial de ese experto? (s/N): ").strip().lower() in ("s", "si", "sí", "y")
            bot.cambiar_experto(nueva_clave, conservar_historial=conservar)
            estado = "conservando" if conservar else "reiniciando"
            print(f"✅ Cambiado a {bot.nombre_experto} ({estado} su historial; "
                  f"{len(bot.conversation)} mensajes).")
            _mostrar_ayuda()  # recordatorio para quien se incorpora tarde
            continue

        # Mensaje normal -> al experto activo
        print(f"\n[{bot.nombre_experto}]: ", end="", flush=True)
        bot.enviar(entrada)


### ▶️ Opción A — Conversar sin `input()` (recomendado en notebooks)

Si el cuadro de `input()` te resulta incómodo en el notebook, crea el bot **una sola vez** y háblale con `bot.enviar(...)`. El historial se conserva porque reutilizas el mismo objeto `bot`. También puedes cambiar de experto con `bot.cambiar_experto(...)`.

> Ejecuta antes, de arriba abajo, las celdas de `EXPERTOS`, `Conversation` y `Chatbot`.


In [7]:
# Crear el bot UNA sola vez (vuelve a ejecutar esta celda para empezar de cero)
bot = Chatbot(EXPERTOS, modelo=MODELO, experto_inicial="programacion")
print(f"Experto inicial: {bot.nombre_experto}")

# 1) Mensaje al experto en programación (mantiene contexto en sucesivas llamadas)
print(f"\n[{bot.nombre_experto}]: ", end="")
bot.enviar("¿Qué es la inyección de dependencias y por qué es útil?")

# 2) Cambiar a marketing REINICIANDO el historial
bot.cambiar_experto("marketing", conservar_historial=False)
print(f"\n[{bot.nombre_experto}]: ", end="")
bot.enviar("Dame 3 ideas de campaña para lanzar una app de finanzas personales.")

# 3) Ver el historial en memoria del experto actual
print("\n\nHistorial actual:", bot.conversation.messages)


La Injección de Dependencias (Dependency Injection – DI) es un patrón de diseño que reduce la coupling entre una parte del sistema (que utiliza una biblioteca o módulo) y las otras partes. En lugar de que la parte utilizando una dependencia tenga que saber cómo crear o obtener su instancia, ésta se le "inyecta" como un parámetro o variable.

**¿Por qué es útil?**

*   **Paralelismo:** Permite separar las responsabilidades en diferentes módulos del sistema, así que puedes probar y desarrollar las partes de código de forma modular con mucho más confianza.
*   **Refactoring sencillo:** Es mucho más fácil modificar una dependencia sin tener que cambiar el código que usa esa dependencia. La única necesidad es cambiar el input del parámetro.
*   **Testabilidad Mejorada:** Es mucho, mucho más fácil de testar y depurar las partes del sistema que dependen de otras, ya que la dependencia no es parte del código real y se puede usar un mock o stub para replicarle su comportamiento en pruebas.
*   

### ▶️ Opción B — CLI interactiva con `run_cli()`

Lanza el bucle con menú, cambio de experto y comandos. En una terminal real se ejecutaría con `python main.py`.


In [8]:
run_cli()


🧠 Chatbot de expertos con Ollama (modelo local)
✅ Ollama operativo y modelo 'gemma3:1b' disponible en local.

Expertos disponibles:
  1. 💻 Experto en Programación  (clave: programacion)
  2. 📈 Experto en Marketing  (clave: marketing)
  3. ⚖️ Experto Jurídico-Legal  (clave: legal)

▶️  Experto activo: 💻 Experto en Programación

Comandos disponibles:
  /cambiar    cambiar de experto (cada experto guarda su propio historial)
  /reiniciar  borrar el historial del experto actual
  /comprobar  re-verificar que Ollama y el modelo siguen disponibles
  /experto    ver el experto activo y el tamaño del historial
  /ayuda      mostrar esta ayuda
  /salir      terminar
Escribe cualquier otro texto para enviarlo como mensaje al experto activo.

[💻 Experto en Programación]: ¡Claro! Aquí tienes un programa "Hola Mundo" en ensamblador x86 (ASLR) - diseñado con una atención al detalle técnica.  Considera esto como un ejemplo de programación en el lenguaje de las computadoras, no un simple 'escrita con 

## 7. (Opcional) Prueba del manejo de errores

Esta celda **no forma parte de la solución entregable**: solo demuestra que el chatbot maneja con elegancia el caso en que Ollama no está disponible. Apuntamos el cliente a un host inexistente para forzar un fallo de conexión; el bot debe informar del error y devolver `None` sin romperse.


In [9]:
# Forzamos un fallo de conexión apuntando a un host donde no hay servicio de Ollama
bot_roto = Chatbot(EXPERTOS, modelo=MODELO, host="http://localhost:1")
print(f"[{bot_roto.nombre_experto}]: ", end="")
resultado = bot_roto.enviar("Hola, ¿me oyes?")
print("Valor devuelto:", resultado)
print("El historial queda limpio tras el fallo:", bot_roto.conversation.messages)


[💻 Experto en Programación]: 
❌ No se pudo contactar con Ollama (ConnectError: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión). ¿Está 'ollama serve' en marcha?
Valor devuelto: None
El historial queda limpio tras el fallo: []
